# Artigo 0008 - LLMs Locais com Ollama

Benchmark de modelos locais servidos pelo Ollama em Docker, sem chave de API e sem custo por token.

## Stack usado
- Servidor: `Ollama` em container Docker (`http://localhost:11434`)
- Modelos de chat: `qwen2.5:0.5b`, `qwen2.5:1.5b` e `llama3.2:1b`
- Modelo de embeddings: `nomic-embed-text`

Pré-requisito: subir o container com `docker compose up -d` e puxar os modelos (ver README do módulo).

In [1]:
from pathlib import Path
import sys
import json

# Faz o notebook funcionar tanto na pasta do artigo quanto via raiz do repositório ou subpasta.
candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "0008_llms_locais_ollama",
    Path.cwd().parent,
    Path.cwd().parent / "0008_llms_locais_ollama",
]
ARTICLE_ROOT = next((p.resolve() for p in candidate_dirs if (p / "src" / "ollama_lab.py").exists()), Path.cwd())
if str(ARTICLE_ROOT) not in sys.path:
    sys.path.insert(0, str(ARTICLE_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.ollama_lab import (
    DEFAULT_BASE_URL,
    DEFAULT_CHAT_MODELS,
    DEFAULT_EMBEDDING_MODEL,
    check_server,
    run_lab,
)

## 1. Healthcheck do servidor local

Antes de medir qualquer coisa, provamos que o servidor existe, responde e tem os modelos baixados. Esse passo substitui o cadastro em plataforma de nuvem: quem garante o ambiente aqui é o Docker na sua máquina.

In [2]:
server = check_server(DEFAULT_BASE_URL)
print(f"Ollama {server['version']} em {DEFAULT_BASE_URL}")
pd.DataFrame(server["models"])

Ollama 0.35.1 em http://localhost:11434


,name,size_bytes
0,llama3.2:1b,1321098329
1,qwen2.5:1.5b,986061892
2,nomic-embed-text:latest,274302450
3,qwen2.5:0.5b,397821319


## 2. Benchmark de chat: latência, TTFT e vazão

O runner envia os mesmos prompts para cada modelo e mede:

- `ttft_ms`: tempo até o primeiro token (a métrica que define a sensação de velocidade)
- `total_ms`: latência total da chamada
- `tokens_por_segundo`: vazão de geração no seu hardware

In [3]:
resultados, resumo, estruturado, embeddings = run_lab(
    base_url=DEFAULT_BASE_URL,
    chat_models=DEFAULT_CHAT_MODELS.split(","),
    embedding_model=DEFAULT_EMBEDDING_MODEL,
    repeat=2,
    attempts=5,
)
resumo

1/5 Verificando servidor Ollama...
   Ollama 0.35.1 com 4 modelos locais
2/5 Benchmark de chat (3 modelos x 6 prompts x 2 repeticoes)...


3/5 Benchmark de saida estruturada (JSON)...


4/5 Laboratorio de embeddings e retrieval...


5/5 Persistindo relatorio e grafico...


,modelo,ttft_medio_ms,total_medio_ms,tokens_por_segundo,tokens_gerados_medios,chamadas
1,qwen2.5:0.5b,62.4,592.7,187.8,73.7,12
2,qwen2.5:1.5b,81.3,320.0,106.1,21.2,12
0,llama3.2:1b,47.5,436.1,89.0,33.3,12


## 3. Onde cada modelo acerta e erra

Nem toda diferença aparece na média. Vale inspecionar a resposta por prompt: modelos menores costumam emendar classificação com geração de texto quando deveriam responder apenas o rótulo.

In [4]:
resultados[["modelo", "prompt_id", "ttft_ms", "total_ms", "tokens_gerados", "tokens_por_segundo", "acertou"]]

,modelo,prompt_id,ttft_ms,total_ms,tokens_gerados,tokens_por_segundo,acertou
0,qwen2.5:0.5b,resumo_curto,51.4,1421.9,119,86.8,True
1,qwen2.5:0.5b,resumo_politica,58.7,164.9,18,167.9,True
2,qwen2.5:0.5b,classificacao_intencao,229.6,269.7,6,146.3,False
3,qwen2.5:0.5b,classificacao_incidente,148.0,178.6,6,187.1,False
4,qwen2.5:0.5b,extracao_campos,109.1,300.2,38,197.9,True
5,qwen2.5:0.5b,explicacao_tecnica,49.5,1457.9,255,181.0,True
6,qwen2.5:0.5b,resumo_curto,10.6,698.6,119,173.0,True
7,qwen2.5:0.5b,resumo_politica,12.8,89.0,18,233.0,True
8,qwen2.5:0.5b,classificacao_intencao,9.7,31.0,6,279.3,False
9,qwen2.5:0.5b,classificacao_incidente,9.6,30.3,6,278.4,False


In [5]:
plt.figure(figsize=(9, 4))
for modelo, grupo in resultados.groupby("modelo"):
    plt.scatter(grupo["total_ms"], grupo["tokens_por_segundo"], label=modelo, s=60)
plt.title("Latência x vazão dos modelos locais")
plt.xlabel("Latência total (ms)")
plt.ylabel("Tokens/s")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

/var/folders/68/sfvst3ln2sl8jbqzrj41gzz40000gn/T/ipykernel_87020/3498219196.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Saída estruturada: o modelo sustenta um planner?

O agente do artigo 0007 depende de JSON válido. Aqui testamos a mesma exigência contra cada modelo local, com `format: "json"` forçado no Ollama.

In [6]:
estruturado

,modelo,tentativas,json_valido,json_valido_pct,tool_correta,tool_correta_pct
0,qwen2.5:0.5b,5,5,100.0,5,100.0
1,qwen2.5:1.5b,5,5,100.0,5,100.0
2,llama3.2:1b,5,0,0.0,0,0.0


## 5. Embeddings locais: retrieval sem nuvem

A mesma instância que serve o chat serve embeddings. O roteamento semântico do artigo 0007 pode, portanto, rodar inteiro no seu computador.

In [7]:
rotas = pd.read_csv(ARTICLE_ROOT / "data" / "embedding_routes.csv")
embeddings

,modelo_embedding,dimensao,documentos,latencia_batch_docs_ms,latencia_batch_queries_ms,taxa_acerto_top1
0,nomic-embed-text,768,8,1278.4,43.7,1.0


In [8]:
rotas

,consulta_id,consulta,documento_top1,score_top1,esperado,acertou
0,q_devolucao,posso devolver um produto sem uso?,politica_devolucao,0.7460,politica_devolucao,True
1,q_fatura,como pego a segunda via do boleto?,segunda_via_fatura,0.7191,segunda_via_fatura,True
2,q_incidente,o sistema esta fora do ar com erro 500,suporte_aplicacao,0.7501,suporte_aplicacao,True
3,q_senha,posso passar minha senha no chat?,seguranca_conta,0.6252,seguranca_conta,True


## 6. Artefatos gerados

- `data/benchmark_resultados.csv` - linha a linha de cada chamada medida
- `data/benchmark_resumo.csv` - consolidação por modelo
- `data/structured_output.csv` - taxa de JSON válido e tool correta
- `data/embedding_routes.csv` - rotas semânticas resolvidas localmente
- `data/benchmark_relatorio.md` - relatório pronto para leitura
- `data/modelos_disponiveis.json` - snapshot do servidor no momento da execução

In [9]:
print((ARTICLE_ROOT / "data" / "benchmark_relatorio.md").read_text(encoding="utf-8"))

# Relatorio do laboratorio local de LLMs

- Servidor: Ollama `0.35.1` em `http://localhost:11434`
- Modelos no container: llama3.2:1b, qwen2.5:1.5b, nomic-embed-text:latest, qwen2.5:0.5b
- Execucao: 36 chamadas de chat, sem chave de API, sem custo por token

## Chat: latencia e vazao por modelo

| modelo | ttft_medio_ms | total_medio_ms | tokens_por_segundo | tokens_gerados_medios | chamadas |
| --- | --- | --- | --- | --- | --- |
| qwen2.5:0.5b | 62.4 | 592.7 | 187.8 | 73.7 | 12 |
| qwen2.5:1.5b | 81.3 | 320.0 | 106.1 | 21.2 | 12 |
| llama3.2:1b | 47.5 | 436.1 | 89.0 | 33.3 | 12 |

## Saida estruturada: JSON valido e tool correta

| modelo | tentativas | json_valido | json_valido_pct | tool_correta | tool_correta_pct |
| --- | --- | --- | --- | --- | --- |
| qwen2.5:0.5b | 5 | 5 | 100.0 | 5 | 100.0 |
| qwen2.5:1.5b | 5 | 5 | 100.0 | 5 | 100.0 |
| llama3.2:1b | 5 | 0 | 0.0 | 0 | 0.0 |

## Embeddings locais (nomic-embed-text)

| modelo_embedding | dimensao | documentos | latencia_batch_

## Próximos passos

1. Repita o benchmark com os modelos que você pretende usar em produção local.
2. Compare o TTFT com o SLA do seu produto antes de escolher o modelo.
3. Use o modelo vencedor como planner no agente do artigo 0007 e rode os evals do artigo 0006.